# Approximate QFT on 6 Qubits

In this Jupyter notebook, we implement AQFT Circuit 1 from Park-Ahn, with $n = 6$ qubits, error $\varepsilon = 0.2$, and cutoff $b = 5$. We take as a given quantum resource that the ancilla state $\lvert \psi_6 \rvert$ has been prepared correctly.

In [1]:
from qrisp import QuantumSession, QuantumVariable, prepare, h, z, s, gidney_adder, QFT

import matplotlib.pyplot as plt

import numpy as np

## Fan Out

The following method applied a fan-out gate as defined in Park-Ahn, which is a sequence of CX gates controlled by a single qubit.

In [2]:
def fan_out(circuit, qubit_index, ancilla_indices):
    """Fan-out operation using CNOT gates.
    
        Input: circuit -> QuantumCircuit object
               qubit_index -> index of the qubit to fan out from
               ancilla_indices -> list of indices of the ancilla qubits to fan out to

        Output: None
    
    """
    circuit.cx(qubit_index, ancilla_indices)

    return None

## AQFT Circuit

In [3]:
n = 6 #number of qubits                         # n = number of qubits we are applying the AQFT to
epsilon = 0.2                                   # epsilon = spectral norm error tolerance for the approximation
b = int(np.ceil(np.log2(n/epsilon)))            # b + 1 = number of ancilla qubits needed
print(f"Number of ancilla qubits needed: {b+1}")

Number of ancilla qubits needed: 6


We initialize the circuit with one dummy register for the purposes of implementing inverse Phase Gradient Transform using a Gidney adder; with one quantum register on $n$ qubits; and one ancilla register on $b + 1$ qubits.

In [4]:
qs = QuantumSession()

quantumRegister = QuantumVariable(n, name='q', qs = qs)          # quantum register of n qubits
ancillaRegister = QuantumVariable(int(b+1), name='a', qs = qs)   # ancilla register of b+1 qubits

In [5]:
# Input state preparation
amplitudes = np.random.randint(0, high=10, size=64, dtype=int)  # generate random amplitudes for the input state
psi = prepare(quantumRegister, amplitudes)                      # prepare the input state on the quantum register based on the generated amplitudes

We assume that we have access to the correct state for ancilla register.

In [6]:
# This is done by preparing the ancilla register in the state |1>^(b+1) using the prepare function from qrisp.
allOnesAmplitudes = np.array([0]*(2**(b+1) - 1) + [1], dtype=float)
allOnes = prepare(ancillaRegister, allOnesAmplitudes)  # prepare the ancilla register in the state |1>^(b+1)

# We apply inverse Quantum Fourier Transform to the all ones state to prepare the ancilla correctly for use with Gidney adder
QFT(ancillaRegister, inv=True, qiskit_endian=False)  # apply the inverse QFT to the ancilla register

<QuantumVariable 'a'>

In [7]:
# Circuit Block 1

h(quantumRegister[0])  # apply Hadamard gate to the first qubit
z(quantumRegister[0])  # apply Z gate to the first qubit
for i in range(1, n):
    s(quantumRegister[i])  # apply S gate to the i-th qubit

gidney_adder(quantumRegister[0:b-1], ancillaRegister)  # apply the Gidney adder

fan_out(qs, quantumRegister[0], quantumRegister[1:b-1])  # apply the fan-out operation to the first qubit and the ancilla register

gidney_adder(quantumRegister[0:b-1], ancillaRegister)  # apply the Gidney adder again

fan_out(qs, quantumRegister[0], quantumRegister[1:b-1])  # apply the fan-out operation to the first qubit and the ancilla register again

In [8]:
# Circuit Block 2

h(quantumRegister[1])  # apply Hadamard gate to the first qubit
s(quantumRegister[1])  # apply S gate to the first qubit

fan_out(qs, quantumRegister[1], quantumRegister[2:b])  # apply the fan-out operation to the first qubit and the ancilla register

gidney_adder(quantumRegister[1:b], ancillaRegister)  # apply the Gidney adder again

fan_out(qs, quantumRegister[1], quantumRegister[2:b])  # apply the fan-out operation to the first qubit and the ancilla register again

In [9]:
# Circuit Block 3

h(quantumRegister[2])  # apply Hadamard gate to the first qubit
s(quantumRegister[2])  # apply S gate to the first qubit

fan_out(qs, quantumRegister[2], quantumRegister[3:n])  # apply the fan-out operation to the first qubit and the ancilla register

gidney_adder(quantumRegister[2:n], ancillaRegister)  # apply the Gidney adder again

fan_out(qs, quantumRegister[2], quantumRegister[3:n])  # apply the fan-out operation to the first qubit and the ancilla register again

In [10]:
# Circuit Block 4

h(quantumRegister[3])  # apply Hadamard gate to the first qubit
s(quantumRegister[3])  # apply S gate to the first qubit

fan_out(qs, quantumRegister[3], quantumRegister[4:n])  # apply the fan-out operation to the first qubit and the ancilla register

gidney_adder(quantumRegister[3:n], ancillaRegister)  # apply the Gidney adder again

fan_out(qs, quantumRegister[3], quantumRegister[4:n])  # apply the fan-out operation to the first qubit and the ancilla register again

In [11]:
# Circuit Block 5

h(quantumRegister[4])  # apply Hadamard gate to the first qubit
s(quantumRegister[4])  # apply S gate to the first qubit

fan_out(qs, quantumRegister[4], quantumRegister[5:n])  # apply the fan-out operation to the first qubit and the ancilla register

gidney_adder(quantumRegister[4:n], ancillaRegister)  # apply the Gidney adder again

fan_out(qs, quantumRegister[4], quantumRegister[5:n])  # apply the fan-out operation to the first qubit and the ancilla register again

In [12]:
# Circuit Block 5

h(quantumRegister[5])  # apply Hadamard gate to the fifth qubit

Qubit(q.5)

In [13]:
# Circuit Block 6

for i in range(n):
    s(quantumRegister[i])  # apply S gate to the i-th qubit

gidney_adder(quantumRegister[-1:-5:-1], ancillaRegister)  # apply the Gidney adder again